In [0]:
# 1. CẤU HÌNH VÀ TẠO SCHEMA SILVER TRONG UNITY CATALOG
CATALOG_NAME = "omni_retail_catalog"
SCHEMA_NAME = "silver"

spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")

print(f"Đang làm việc tại: {CATALOG_NAME}.{SCHEMA_NAME}")

# 2. XỬ LÝ DỮ LIỆU ĐƠN HÀNG SILVER (fnd_sales_items)
# - Dùng try_cast để xử lý an toàn chuỗi rỗng '' trên order_datetime, qty và price
# - Unnest mảng ordered_products và lọc bỏ bản ghi không hợp lệ
from pyspark.sql import functions as F

bronze_sales_table = f"{CATALOG_NAME}.bronze.sales_orders"
target_silver_sales = f"{CATALOG_NAME}.{SCHEMA_NAME}.fnd_sales_items"

df_bronze_sales = spark.table(bronze_sales_table)

df_silver_sales = (
    df_bronze_sales
    .filter(F.col("order_number").isNotNull() & F.col("customer_id").isNotNull())
    .select(
        F.col("order_number").cast("string").alias("order_id"),
        # Sử dụng try_cast để tránh lỗi khi order_datetime là chuỗi rỗng ''
        F.expr("to_timestamp(from_unixtime(try_cast(order_datetime AS BIGINT)))").alias("order_timestamp"),
        F.col("customer_id").cast("string").alias("customer_id"),
        F.explode("ordered_products").alias("item")
    )
    .select(
        "order_id",
        "order_timestamp",
        F.to_date("order_timestamp").alias("order_date"),
        "customer_id",
        F.col("item.id").cast("string").alias("product_id"),
        F.col("item.name").cast("string").alias("product_name"),
        # try_cast bảo vệ chống lỗi ép kiểu cho quantity và unit_price
        F.expr("try_cast(item.qty AS INT)").alias("quantity"),
        F.expr("try_cast(item.price AS DECIMAL(18,2))").alias("unit_price")
    )
    # Lọc bỏ các bản ghi có timestamp bị null do dữ liệu nguồn bẩn
    .filter(
        F.col("order_timestamp").isNotNull() & 
        F.col("product_id").isNotNull() &
        F.col("quantity").isNotNull() & (F.col("quantity") > 0) &
        F.col("unit_price").isNotNull() & (F.col("unit_price") >= 0)
    )
    .withColumn("line_total", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .dropDuplicates(["order_id", "product_id"])
)

(
    df_silver_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_silver_sales)
)

print(f"Hoàn thành Silver Sales Items: {spark.table(target_silver_sales).count():,} bản ghi.")

# 3. XỬ LÝ KHÁCH HÀNG THEO CHUẨN SCD TYPE 2 (dim_customer_scd2)
bronze_cust_table = f"{CATALOG_NAME}.bronze.customers"
target_silver_cust = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_customer_scd2"

df_bronze_cust = spark.table(bronze_cust_table)

df_silver_cust = (
    df_bronze_cust
    .filter(F.col("customer_id").isNotNull())
    .select(
        F.col("customer_id").cast("string"),
        F.col("customer_name").cast("string"),
        # Khởi tạo email chuẩn PII từ customer_id phục vụ Data Governance / Masking
        F.lower(F.concat(F.col("customer_id"), F.lit("@retailorg.com"))).alias("email"),
        F.col("city").cast("string"),
        F.col("state").cast("string")
    )
    .dropDuplicates(["customer_id"])
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

(
    df_silver_cust.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_silver_cust)
)

print(f"Hoàn thành Silver Customers SCD2: {spark.table(target_silver_cust).count():,} khách hàng.")